1. 전처리를 잘 해줬지만 답변이 나오지 않는 이유는 질문이 잘못되었기 때문.
2. 직장인과 거주자의 연관성이 부족했음.
3. 그래서 직장인을 거주자로 바꿔 줄 수 있는 LCEL(LangChain Expression Language)

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,    
)

loader = Docx2txtLoader('./tax_with_markdown.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [2]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [5]:
import os
import time

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pinecone_api_key = os.environ.get("PINECONE_API_KEY")
index_name='tax-markdown-index'

pc=Pinecone(api_key=pinecone_api_key)

#database=PineconeVectorStore.from_documents(document_list,embedding,index_name=index_name)

# 먼저 database 객체 생성
database = PineconeVectorStore(index_name=index_name, embedding=embedding)

# documents를 나누어 업로드 (예: 100개씩 배치 처리)
batch_size = 100
for i in range(0, len(document_list), batch_size):
    batch = document_list[i:i + batch_size]
    database.add_documents(batch)

In [6]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'

In [ ]:
retriever = database.as_retriever(search_kwargs={'k': 4})
retriever.invoke(query)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

from langchain import hub

prompt = hub.pull("rlm/rag-prompt")

In [9]:
#prompt
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm, 
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

In [10]:
ai_message = qa_chain.invoke({"query": query})

In [ ]:
ai_message


In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요
    사전: {dictionary}
    
    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()
tax_chain = {"query": dictionary_chain} | qa_chain

In [ ]:
new_question = dictionary_chain.invoke({"question": query})
query


In [ ]:
new_question

In [15]:
ai_response = tax_chain.invoke({"question": query})
ai_response

{'query': '연봉 5천만원인 거주자의 소득세는 얼마인가요?',
 'result': '연봉 5천만 원인 거주자의 소득세는 840,000원 + (3,600만 원 × 15%) = 6,240,000원입니다.'}